# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/venkat-vipul/flyrank_ml/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

## My Lane as an ML Task

I chose **Refresh / Content Opportunity Scoring**, which is primarily a **ranking and scoring** machine learning task. The goal is not simply to classify whether a page is good or bad, but to rank content pages according to how strongly they should be considered for human review. Each page receives a score based on observable search and content signals, allowing reviewers to prioritize the most important pages first. This ranking supports efficient decision-making when review time and resources are limited.

In [11]:
import pandas as pd

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")

print("ML Task: Ranking / Scoring")
print(f"Dataset contains {len(df):,} content pages.")
print(df[["content_id", "impressions_90d", "trend_direction"]].head())

ML Task: Ranking / Scoring
Dataset contains 30,000 content pages.
             content_id  impressions_90d trend_direction
0  content_304f48230142             3803            down
1  content_a1fb4e703a9e            15320            down
2  content_9aa793d4d895            12581            down
3  content_331d6c4de07b            11751          stable
4  content_d99b7a2d90ca            19140            down


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

## Target or Proxy

For this starter project, I will use **whether a page is currently declining** as the target proxy. This is represented by the "trend_direction" field, where pages labeled "down" are treated as positive review candidates.

This is a **proxy label** rather than the ideal real-world target. The long-term goal would be to predict future content that should be reviewed before its performance declines. However, in the starter dataset, the current trend provides a measurable and transparent target for developing and evaluating the ranking system.

In [12]:
target = df["trend_direction"]

print("Target column:")
print(target.head())

print("\nTarget distribution:")
print(target.value_counts())

Target column:
0      down
1      down
2      down
3    stable
4      down
Name: trend_direction, dtype: object

Target distribution:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64


## 3. Success metric

*One metric you can defend. What number means 'good'?*



The primary success metric for this project is **Precision@50**. Since the goal is to help reviewers prioritize a limited number of pages, the quality of the highest-ranked recommendations is more important than overall accuracy. Precision@50 measures how many of the top 50 recommended pages are actually relevant according to the chosen target proxy. A higher Precision@50 indicates that reviewers are more likely to spend their limited time on genuinely useful review candidates.

In [13]:
print("Chosen evaluation metric: Precision@50")

top_candidates = (
    df.sort_values("impressions_90d", ascending=False)
      .head(50)
)

print(f"Example ranked review queue: {len(top_candidates)} pages")

top_candidates[
    ["content_id", "impressions_90d", "trend_direction"]
].head()

Chosen evaluation metric: Precision@50
Example ranked review queue: 50 pages


,content_id,impressions_90d,trend_direction
6653,content_5fe46e04994d,517715,down
17812,content_aaef01a50def,517109,stable
26844,content_8c19996aa890,509252,down
19636,content_2cb567c3c89b,497727,up
21819,content_4c36c775b818,463103,down


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

## The Unit of Analysis

The unit of analysis for this project is **one content page**. Each row in the dataset represents a single anonymized content page with observable search and content signals, such as impressions, clicks, CTR, average position, content age, and trend direction. The model produces one score for each page, allowing pages to be ranked according to their priority for human review.

In [14]:
sample_df = df[
    [
        "content_id",
        "impressions_90d",
        "clicks_90d",
        "ctr",
        "avg_position",
        "content_age_days",
        "trend_direction",
    ]
].head()

print("Unit of analysis: One row = One content page")
sample_df

Unit of analysis: One row = One content page


,content_id,impressions_90d,clicks_90d,ctr,avg_position,content_age_days,trend_direction
0,content_304f48230142,3803,29,0.76,10.6,187,down
1,content_a1fb4e703a9e,15320,7,0.05,20.3,445,down
2,content_9aa793d4d895,12581,11,0.09,36.5,141,down
3,content_331d6c4de07b,11751,58,0.49,6.2,463,stable
4,content_d99b7a2d90ca,19140,24,0.13,44.0,263,down


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*



A simple rule-based system might recommend reviewing pages using fixed thresholds, such as content age or impressions alone. However, content performance depends on multiple interacting signals, including impressions, clicks, CTR, average position, content age, and trend direction. These relationships are difficult to capture with a single set of if-else rules.

A machine learning model can learn patterns from many features simultaneously and produce a ranked list of review candidates based on the combined evidence. This allows the system to support more consistent and data-driven prioritization while still leaving the final decision to a human reviewer.

In [15]:
rule_candidates = df[
    (df["impressions_90d"] > 10000) &
    (df["content_age_days"] > 180)
]

print(f"Pages matching a simple fixed rule: {len(rule_candidates):,}")

rule_candidates[
    [
        "content_id",
        "impressions_90d",
        "content_age_days",
        "trend_direction"
    ]
].head()

Pages matching a simple fixed rule: 2,167


,content_id,impressions_90d,content_age_days,trend_direction
1,content_a1fb4e703a9e,15320,445,down
3,content_331d6c4de07b,11751,463,stable
4,content_d99b7a2d90ca,19140,263,down
10,content_d8ee6cc6d642,20919,329,stable
16,content_78bd1d4a1d4d,13848,307,down


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.